# Road accident Severity - Model Training

## 1. Data Preparation

In [1]:
# import libs
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
# ML
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings("ignore")

In [2]:
PROJECT_DIR = Path.cwd().resolve().parents[1]
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

### 1.1. Load Processed Dataset

In [3]:
dataset_file_path = PROCESSED_DATA_DIR.resolve() / "accident_severity_dataset_processed.csv"
df = pd.read_csv(dataset_file_path, low_memory=False)

print(df.shape)
df.head()

(440337, 37)


,catu,grav,sexe,trajet,locp,etatp,catv,obs,obsm,choc,...,surf,infra,situ,vma,year,age,age_unknown,hour,has_safety_equipment,grav_ord
0,1,3,1,5,-1,-1,motorcycle,none,2,1,...,1,0,1,50,2022,14,0,16,1,2.0
1,1,1,1,5,-1,-1,car,none,2,2,...,1,0,1,50,2022,74,0,16,1,0.0
2,1,4,1,9,0,-1,car,none,2,8,...,1,0,1,50,2022,34,0,8,1,1.0
3,1,1,1,4,0,-1,truck,none,2,1,...,1,0,1,50,2022,52,0,8,1,0.0
4,1,1,1,0,-1,-1,car,none,2,1,...,1,5,1,50,2022,20,0,17,1,0.0


### 1.2. Dataset Overview

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 440337 entries, 0 to 440336
Data columns (total 37 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   catu                  440337 non-null  int64  
 1   grav                  440337 non-null  int64  
 2   sexe                  440337 non-null  int64  
 3   trajet                440337 non-null  int64  
 4   locp                  440337 non-null  int64  
 5   etatp                 440337 non-null  int64  
 6   catv                  440337 non-null  str    
 7   obs                   440337 non-null  str    
 8   obsm                  440337 non-null  int64  
 9   choc                  440337 non-null  int64  
 10  manv                  440337 non-null  int64  
 11  motor                 440337 non-null  int64  
 12  jour                  440337 non-null  int64  
 13  mois                  440337 non-null  int64  
 14  lum                   440337 non-null  int64  
 15  agg        

In [5]:
df.describe(include="all").transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
catu,440337.0,NaN,NaN,NaN,1.325369,0.608368,1.0,1.0,1.0,2.0,3.0
grav,440337.0,NaN,NaN,NaN,2.511127,1.382103,-1.0,1.0,3.0,4.0,4.0
sexe,440337.0,NaN,NaN,NaN,1.269587,0.565101,-1.0,1.0,1.0,2.0,2.0
trajet,440337.0,NaN,NaN,NaN,3.133543,2.785088,-1.0,0.0,4.0,5.0,9.0
locp,440337.0,NaN,NaN,NaN,-0.216357,1.25219,-1.0,-1.0,0.0,0.0,9.0
etatp,440337.0,NaN,NaN,NaN,-0.819561,0.631314,-1.0,-1.0,-1.0,-1.0,3.0
catv,440337,10,car,274844,NaN,NaN,NaN,NaN,NaN,NaN,NaN
obs,440337,5,none,373187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
obsm,440337.0,NaN,NaN,NaN,1.611868,1.229436,-1.0,1.0,2.0,2.0,9.0
choc,440337.0,NaN,NaN,NaN,2.847533,2.393336,-1.0,1.0,2.0,4.0,9.0


In [6]:
df.nunique()

catu                      3
grav                      5
sexe                      3
trajet                    8
locp                     11
etatp                     4
catv                     10
obs                       5
obsm                      8
choc                     11
manv                     28
motor                     8
jour                     31
mois                     12
lum                       6
agg                       2
int                      10
atm                      10
col                       8
catr                      8
v1                        4
circ                      5
nbv                      16
vosp                      5
prof                      5
plan                      5
larrout                 137
surf                     10
infra                    11
situ                      8
vma                      39
year                      3
age                     108
age_unknown               2
hour                     24
has_safety_equipment

In [ ]:
# missing values
df.isna().sum()[df.isna().sum() > 0]

grav_ord    387
dtype: int64

## 2. Goal


- __Hypothesis__: Can accident serverity be prdicted only using environmental and road conditions?
 
- __Target__: grav_order

- __Predictors__: environmental vars + road characteristics

## 3. Feature Selection

In [9]:
# Selected features

features = [
    "atm",# weather
    "surf",# road surface
    "lum",# lighting
    "infra",# infrastructure
    "situ",# accident location
    "plan",# road profile
    "catr",# road category
    "agg",# urban / rural
    "vma",# speed limit
    "hour",# accident time
    "mois",# seasonality

    # user information
    "age",
    "age_unknown",
]

In [15]:
# features and target
X = df[features]
y = df["grav_ord"]
y.dtypes

dtype('float64')

In [16]:
# grav_ord contain NaN -> grav = -1 -> grav_ord = NaN
# grav_ord: 387 -> can be removed
mask = y.notna()

X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].astype(int).reset_index(drop=True)

print(X.shape)
print(y.shape)

(439950, 13)
(439950,)


In [17]:
# features types
categorical_features = [
    "atm",
    "surf",
    "lum",
    "infra",
    "situ",
    "plan",
    "catr",
    "agg",
]

numerical_features = [
    "vma",
    "hour",
    "mois",
    "age",
    "age_unknown",
]

In [ ]:
# correlation mat
# most variables are ordinal or categorical encoded and not continue vars: so this corr is only used for exploratory
corr = X.join(y).corr(numeric_only=True)
corr

,atm,surf,lum,infra,situ,plan,catr,agg,vma,hour,mois,age,age_unknown,grav_ord
atm,1.000000,0.221900,0.010152,0.023681,0.013051,0.028488,0.003962,-0.016247,0.011153,-0.030865,0.034207,0.014510,-0.009198,0.013982
surf,0.221900,1.000000,0.081347,0.024949,0.038150,0.079770,-0.011909,-0.052301,0.041278,-0.023502,0.026360,-0.019394,-0.006385,0.025004
lum,0.010152,0.081347,1.000000,0.024279,0.018953,-0.020628,0.045492,0.121696,-0.065797,0.055904,0.061656,-0.140248,0.051498,0.005486
infra,0.023681,0.024949,0.024279,1.000000,0.092608,0.018443,0.047738,0.066226,-0.077949,-0.002297,0.004546,0.017433,0.000143,-0.007978
situ,0.013051,0.038150,0.018953,0.092608,1.000000,0.055477,0.090234,0.023371,-0.056940,-0.031113,-0.000781,-0.001874,0.003810,0.087276
plan,0.028488,0.079770,-0.020628,0.018443,0.055477,1.000000,-0.036089,-0.174930,0.093547,-0.020341,0.006013,-0.007375,-0.021333,0.101260
catr,0.003962,-0.011909,0.045492,0.047738,0.090234,-0.036089,1.000000,0.491673,-0.494616,0.024433,-0.015979,0.009178,0.031660,-0.019430
agg,-0.016247,-0.052301,0.121696,0.066226,0.023371,-0.174930,0.491673,1.000000,-0.640081,0.026794,-0.019586,-0.004007,0.056497,-0.142391
vma,0.011153,0.041278,-0.065797,-0.077949,-0.056940,0.093547,-0.494616,-0.640081,1.000000,-0.033841,0.019969,-0.012900,-0.043073,0.077850
hour,-0.030865,-0.023502,0.055904,-0.002297,-0.031113,-0.020341,0.024433,0.026794,-0.033841,1.000000,0.001624,-0.005711,-0.001342,-0.029113


In [19]:
fig = px.imshow(
    corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Correlation Matrix"
)
fig.show()

In [ ]:
# corr with target
# agg: -0.14, vma: ~0.08 -> vma is reduced in agg which reduce accident severity. outside agg vma is high so accident are more grav
corr_target = corr["grav_ord"].sort_values(ascending=False)
corr_target

grav_ord       1.000000
plan           0.101260
situ           0.087276
vma            0.077850
surf           0.025004
age            0.017546
atm            0.013982
lum            0.005486
mois           0.004150
infra         -0.007978
catr          -0.019430
hour          -0.029113
age_unknown   -0.142178
agg           -0.142391
Name: grav_ord, dtype: float64

In [21]:
px.bar(corr_target, title="Correlation Matrix with accident severity")

In [22]:
severity_dist = y.value_counts().sort_index().rename_axis("Severity").reset_index(name="Count")

severity_dist

,Severity,Count
0,0,188559
1,1,174671
2,2,65446
3,3,11274


In [ ]:
# https://plotly.com/python/categorical-axes/
fig = px.bar(
    severity_dist,
    x="Severity",
    y="Count",
    text="Count",
    title="Target Distribution"
)

fig.update_xaxes(type="category")
fig.show()

docs:
- [pandas docs](https://pandas.pydata.org/docs/user_guide/index.html)
- [plotly](https://plotly.com/python/)
- [scikit-learn - data leakage](https://scikit-learn.org/stable/common_pitfalls.html)